# Cardenality Sales Data — Cleaning Pipeline

This notebook loads six raw sheets from `Cardenality_Sales_Data.xlsx`, explores them to find data quality issues, cleans and standardizes the data, and exports clean CSVs for downstream analysis.

## 1. Load Raw Data

Load all six sheets exactly as they come from the source workbook, with no column renaming yet, so the exploration below reflects the raw data as delivered.

In [93]:
import pandas as pd

In [94]:
df_customers = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Customers Sheet', header=1)

In [95]:
df_stores = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Store Locations Sheet', header=0)

In [96]:
df_products = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Products Sheet', header=1)

In [97]:
df_salesteam = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Sales Team Sheet', header=1)

In [98]:
df_regions = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Regions Sheet', header=1)

In [99]:
df_sales = pd.read_excel('Cardenality_Sales_Data.xlsx', sheet_name='Sales Orders Sheet', header=0)

## 2. Exploratory Data Analysis

Before touching anything, check shape, dtypes, missing values, duplicates, and category values across all six sheets. This is where the issues fixed below were found: malformed `Sales Channel` labels, missing `StateCode` values, a missing/corrupted `OrderDate`, missing `Discount Applied` values, and out-of-range order quantities.

In [100]:
sheets = {
    'Customers': df_customers,
    'Products': df_products,
    'SalesTeam': df_salesteam,
    'Regions': df_regions,
    'Stores': df_stores,
    'Sales Orders': df_sales
}

for name, df in sheets.items():
    print(f'================== {name} ==================')
    print(f'Shape: {df.shape}')
    print()
    print('Data Types:')
    print(df.dtypes)
    print()
    print('Missing Values:')
    print(df.isnull().sum())
    print()
    print('Duplicates:', df.duplicated().sum())
    print()
    print('Statistical Summary:')
    print(df.describe())
    print()
    for col in df.select_dtypes(include='object').columns:
        print(f'Unique values in {col}:')
        print(df[col].value_counts().head(10))
        print()

================== Customers ==================
Shape: (50, 2)

Data Types:
_CustomerID        int64
Customer Names    object
dtype: object

Missing Values:
_CustomerID       0
Customer Names    0
dtype: int64

Duplicates: 0

Statistical Summary:
       _CustomerID
count     50.00000
mean      25.50000
std       14.57738
min        1.00000
25%       13.25000
50%       25.50000
75%       37.75000
max       50.00000

Unique values in Customer Names:
Customer Names
Avon Corp        1
Ascend Ltd       1
Dharma Ltd       1
Apotheca, Ltd    1
S.S.S. Group     1
Uriel Group      1
OHTA'S Corp      1
Trigen           1
OUR Ltd          1
Amylin Group     1
Name: count, dtype: int64

================== Products ==================
Shape: (47, 2)

Data Types:
_ProductID       int64
Product Name    object
dtype: object

Missing Values:
_ProductID      0
Product Name    0
dtype: int64

Duplicates: 0

Statistical Summary:
       _ProductID
count   47.000000
mean    24.000000
std     13.711309
min   

## 3. Data Cleaning

With the issues identified, clean the data step by step: rename columns for consistency, fix mislabeled categories, backfill missing values, derive metrics, and remove invalid rows.

### 3.1 Rename Columns

Standardize column names across all sheets. See the rename tracker below for the full mapping.

## Column Rename Tracker

**Customers Sheet**
- _CustomerID → CustomerId
- Customer Names → CustomerName

**Store Locations Sheet**
- _StoreID → StoreId
- City Name → CityName
- Household Income → HouseholdIncome
- Median Income → MedianIncome
- Land Area → LandArea
- Water Area → WaterArea
- Time Zone → TimeZone

**Products Sheet**
- _ProductID → ProductId
- Product Name → ProductName

**Sales Team Sheet**
- _SalesTeamID → SalesTeamId
- Sales Team → SalesTeam

**Regions Sheet**
- (no renaming needed — column names were already clean)

**Sales Orders Sheet**
- _CustomerID → CustomerId
- _StoreID → StoreId
- _ProductID → ProductId
- _SalesTeamID → SalesTeamId
- Order Quantity → OrderQuantity
- Discount Applied → DiscountApplied
- Unit Price → UnitPrice
- Unit Cost → UnitCost
- Sales Channel → SalesChannel

In [101]:
df_customers = df_customers.rename(columns={
    '_CustomerID': 'CustomerId',
    'Customer Names': 'CustomerName'
})

In [102]:
df_stores = df_stores.rename(columns={
    '_StoreID': 'StoreId',
    'City Name': 'CityName',
    'Household Income': 'HouseholdIncome',
    'Median Income': 'MedianIncome',
    'Land Area': 'LandArea',
    'Water Area': 'WaterArea',
    'Time Zone': 'TimeZone'
})

In [103]:
df_products = df_products.rename(columns={
    '_ProductID': 'ProductId',
    'Product Name': 'ProductName'
})

In [104]:
df_salesteam = df_salesteam.rename(columns={
    '_SalesTeamID': 'SalesTeamId',
    'Sales Team': 'SalesTeam'
})

In [105]:
df_sales = df_sales.rename(columns={
    'Sales Channel': 'SalesChannel',
    '_SalesTeamID': 'SalesTeamId',
    '_CustomerID': 'CustomerId',
    '_StoreID': 'StoreId',
    '_ProductID': 'ProductId',
    'Order Quantity': 'OrderQuantity',
    'Discount Applied': 'DiscountApplied',
    'Unit Price': 'UnitPrice',
    'Unit Cost': 'UnitCost',
})

In [106]:
for name, df in {'Customers': df_customers, 'Stores': df_stores, 'Products': df_products,
                  'SalesTeam': df_salesteam, 'Regions': df_regions, 'Sales Orders': df_sales}.items():
    print(name, '->', list(df.columns))

Customers -> ['CustomerId', 'CustomerName']
Stores -> ['StoreId', 'CityName', 'County', 'StateCode', 'State', 'Type', 'Latitude', 'Longitude', 'AreaCode', 'Population', 'HouseholdIncome', 'MedianIncome', 'LandArea', 'WaterArea', 'TimeZone']
Products -> ['ProductId', 'ProductName']
SalesTeam -> ['SalesTeamId', 'SalesTeam', 'Region']
Regions -> ['StateCode', 'State', 'Region']
Sales Orders -> ['OrderNumber', 'SalesChannel', 'WarehouseCode', 'ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate', 'CurrencyCode', 'SalesTeamId', 'CustomerId', 'StoreId', 'ProductId', 'OrderQuantity', 'DiscountApplied', 'UnitPrice', 'UnitCost']


### 3.2 Fix Sales Channel Labels

`SalesChannel` in the raw sales orders sheet contained corrupted values (`Whole#_sale`, `On line`, `In___Store`, `Dis  _tributor`). Standardize them to their intended labels.

In [107]:
df_sales['SalesChannel'] = df_sales['SalesChannel'].replace({
    'Whole#_sale': 'Wholesale',
    'On line': 'Online',
    'In___Store': 'In-Store',
    'Dis  _tributor': 'Distributor'
})

In [108]:
print(df_sales['SalesChannel'].value_counts())

SalesChannel
In-Store       3298
Online         2425
Distributor    1375
Wholesale       893
Name: count, dtype: int64


### 3.3 Backfill Missing StateCode in Store Locations

Some stores are missing `StateCode`. Merge in the `Regions` sheet on `State` to backfill it.

In [109]:
df_merged = df_stores.merge(df_regions[['State', 'StateCode']], on='State', how='left')
df_merged['StateCode_x'] = df_merged['StateCode_x'].fillna(df_merged['StateCode_y'])

In [110]:
df_merged = df_merged.rename(columns={'StateCode_x': 'StateCode'})
df_merged = df_merged.drop(columns=['StateCode_y'])
df_stores_clean = df_merged

In [111]:
print(df_stores_clean[['StateCode', 'State']].head(10))

  StateCode     State
0        AL   Alabama
1        AL   Alabama
2        AL   Alabama
3        AL   Alabama
4        AR  Arkansas
5        AZ   Arizona
6        AZ   Arizona
7        AZ   Arizona
8        AZ   Arizona
9        AZ   Arizona


### 3.4 Fix Sales Order Dates and Discounts

`OrderDate` contained four problematic values — `09haw`, `asdc`, and `7683-08-12` were corrupted entries coerced to NaT by `errors='coerce'`, and one row was genuinely `blank`. All four were backfilled from `ProcuredDate`.

In [112]:
df_sales['OrderDate'] = pd.to_datetime(df_sales['OrderDate'], errors='coerce')
df_sales['OrderDate'] = df_sales['OrderDate'].fillna(df_sales['ProcuredDate'])

In [113]:
df_sales['DiscountApplied'] = df_sales['DiscountApplied'].fillna(0)

### 3.5 Derive Revenue, Profit, and Delivery Metrics

Compute `TotalRevenue`, `TotalProfit`, and `DeliveryDays` from the cleaned order fields.

In [114]:
df_sales['TotalRevenue'] = (df_sales['OrderQuantity'] * df_sales['UnitPrice']) * (1 - df_sales['DiscountApplied'])
df_sales['TotalProfit'] = df_sales['TotalRevenue'] - (df_sales['UnitCost'] * df_sales['OrderQuantity'])
df_sales['DeliveryDays'] = (df_sales['DeliveryDate'] - df_sales['OrderDate']).dt.days

In [115]:
print(df_sales.head())

   OrderNumber SalesChannel WarehouseCode ProcuredDate  OrderDate   ShipDate  \
0  SO - 000101     In-Store  WARE-UHY1004   2017-12-31 2018-05-31 2018-06-14   
1  SO - 000102       Online  WARE-NMK1003   2017-12-31 2018-05-31 2018-06-22   
2  SO - 000103  Distributor  WARE-UHY1004   2017-12-31 2018-05-31 2018-06-21   
3  SO - 000104    Wholesale  WARE-NMK1003   2017-12-31 2018-05-31 2018-06-02   
4  SO - 000105  Distributor  WARE-NMK1003   2018-04-10 2018-05-31 2018-06-16   

  DeliveryDate CurrencyCode  SalesTeamId  CustomerId  StoreId  ProductId  \
0   2018-06-19          USD            6          15      259         12   
1   2018-07-02          USD           14          20      196         27   
2   2018-07-01          USD           21          16      213         16   
3   2018-06-07          USD           28          48      107         23   
4   2018-06-26          USD           22          49      111         26   

   OrderQuantity  DiscountApplied  UnitPrice  UnitCost  TotalR

### 3.6 Remove Invalid Order Quantities

Drop rows where `OrderQuantity` falls outside a realistic 1–999 range.

In [131]:
df_sales = df_sales[(df_sales['OrderQuantity'] >= 1) & (df_sales['OrderQuantity'] < 1000)].copy()

In [117]:
print(df_sales['OrderQuantity'].min())

1.0


### 3.7 Investigate Negative-Profit Orders

Check orders where cost exceeds discounted revenue, to decide whether they're legitimate or a data issue.

In [118]:
print(df_sales[df_sales['TotalProfit'] < 0][['OrderNumber', 'OrderQuantity', 'UnitPrice', 'UnitCost', 'DiscountApplied', 'TotalRevenue', 'TotalProfit']])

       OrderNumber  OrderQuantity  UnitPrice  UnitCost  DiscountApplied  \
13     SO - 000114            6.0      254.6   216.410             0.15   
20     SO - 000121            4.0     2278.0  1754.060             0.40   
58     SO - 000159            1.0      857.6   686.080             0.40   
59     SO - 000160            5.0     3892.7  2880.598             0.30   
88     SO - 000189            2.0     3953.0  3360.050             0.40   
...            ...            ...        ...       ...              ...   
7909  SO - 0008010            6.0     3912.8  2856.344             0.30   
7915  SO - 0008016            8.0      201.0   166.830             0.20   
7936  SO - 0008037            4.0     1715.2  1166.336             0.40   
7938  SO - 0008039            4.0      837.5   711.875             0.20   
7968  SO - 0008069            8.0     6505.7  5139.503             0.40   

      TotalRevenue   TotalProfit  
13         1298.46 -2.273737e-13  
20         5467.20 -1.549040e

### 3.8 Round Revenue and Profit

Round `TotalRevenue` and `TotalProfit` to 2 decimal places for reporting.

In [132]:
df_sales['TotalRevenue'] = df_sales['TotalRevenue'].round(2)
df_sales['TotalProfit'] = df_sales['TotalProfit'].round(2)

In [133]:
print((df_sales['TotalProfit'] < 0).sum())

352


### 3.9 Check for Late Deliveries

Flag any orders with a `DeliveryDays` value greater than 60.

In [55]:
print(df_sales[df_sales['DeliveryDays'] > 60][['OrderNumber', 'OrderDate', 'DeliveryDate', 'DeliveryDays']])

       OrderNumber  OrderDate DeliveryDate  DeliveryDays
313    SO - 000414 2018-04-10   2018-08-01           113
567    SO - 000668 2018-04-10   2018-08-23           135
3649  SO - 0003750 2019-05-15   2019-08-14            91
5129  SO - 0005230 2019-08-23   2020-02-16           177


## 4. Export to CSV

Save all six cleaned DataFrames to CSV for use in downstream analysis and dashboards.

In [ ]:
df_customers.to_csv('customers_clean.csv', index=False)
df_products.to_csv('products_clean.csv', index=False)
df_salesteam.to_csv('sales_team_clean.csv', index=False)
df_regions.to_csv('regions_clean.csv', index=False)
df_stores_clean.to_csv('stores_clean.csv', index=False)
df_sales.to_csv('sales_orders_clean.csv', index=False)